# Secuencia paso a paso del chatbot basado en reglas

## Chatbot DOKi

Este chatbot es un chatbot basado en reglas con la temática de salud mental para los/las estudiantes universitarios
para la escuela de TecMilenio (Campus Toluca - Metepec. Edo.Mex) por lo que las entidades quedan restringidas
únicamente a esta institución.

# Importación de librerias necesarias

In [ ]:
from nltk import re, SnowballStemmer
from pathlib import Path
import unicodedata
import nltk
import json

## ¡Importante!

#### Se debe descargar antes el punkt_tab para poder realizar el tokenizazador y el stem en español

In [ ]:
%pip install --upgrade certifi

In [ ]:
import os
import certifi

os.environ['SSL_CERT_FILE'] = certifi.where()
nltk.download('punkt_tab')

## Importación de reglas del json 

In [ ]:
# Importa las relgas desde el json
ruta_reglas = Path('reglas.json').with_name('reglas.json')
reglas = None

## Carga de las reglas en una variable e inicialización del stemmer

In [ ]:
# Crea un diccionario con las reglas del json para no modificar el código
with ruta_reglas.open(encoding='utf-8') as archivo:
    reglas = json.load(archivo)

# Inicialización del stemmer
stemmer = SnowballStemmer('spanish')

# Función auxiliar para validar strings

In [ ]:
# Verifica que el texto sea válido, no sea nulo o esté vacío
def __texto_valido(texto: str) -> bool:
    return texto is not None and bool(re.search(r"\S", texto))

# 1.- Normalizar el texto

In [ ]:
def normalizar_texto(texto: str | None) -> str:
    """
    Normaliza el texto, quitando los carácteres especiales, puntuaciones
    o inlcuso espacios duplicados, proceso antes de tokenizar
    """
    if not __texto_valido(texto):
        raise ValueError('No se pudo normalizar el texto porque el texto está vacío')
    
    texto = texto.lower().replace('ñ', '\0')
    texto = unicodedata.normalize('NFD', texto)
    texto = "".join(
        caracter
        for caracter in texto
        if unicodedata.category(caracter) != 'Mn'
    )
    texto = texto.replace('\0', 'ñ')
    texto = ''.join(
        caracter
        for caracter in texto
        if caracter.isalnum() or caracter.isspace()
    )

    return ' '.join(texto.split())

# 2.- Tokenizar texto y stems

In [ ]:
def tokenizar_y_stem(texto: str | None) -> tuple[list[str], list[str]]:
    texto_normalizado = normalizar_texto(texto)
    tokens = nltk.word_tokenize(texto_normalizado, language='spanish')
    stems = [stemmer.stem(token) for token in tokens]

    return tokens, stems

tokenizar_y_stem('Hola, como estas')

# 3.- Detectar intenciones

In [ ]:
def detectar_intencion(texto: str | None) -> tuple[str, int]:
    tokens, stems = tokenizar_y_stem(texto)
    stems_mensaje = set(stems)
    candidatas = []
    umbral = reglas['configuracion']['umbral_minimo']

    for intencion in reglas['intenciones']:
        stems_clave = {
            stemmer.stem(token)
            for palabra in intencion['palabras_clave']
            for token in normalizar_texto(palabra).split()
        }
        puntuacion = len(stems_mensaje & stems_clave)

        for patron in intencion['patrones']:
            tokens_patron = normalizar_texto(patron).split()
            longitud_patron = len(tokens_patron)
            if any(
                tokens[indice:indice + longitud_patron] == tokens_patron
                for indice in range(len(tokens) - longitud_patron + 1)
            ):
                puntuacion += 1

        if puntuacion >= umbral:
            candidatas.append(
                (intencion['prioridad'], puntuacion, intencion['nombre'])
            )

    if not candidatas:
        return 'fallback', 0

    prioridad, puntuacion, nombre = min(
        candidatas,
        key=lambda candidata: (candidata[0], -candidata[1]),
    )
    return nombre, puntuacion

detectar_intencion("ya no puedo más")

# 4.- Detectar entidades

En este apartado se detectan las posibles entidades que el usuario ingresó en el texto

In [ ]:
def detectar_entidades(texto: str | None) -> dict[str : str]:
    tokens, stems = tokenizar_y_stem(texto)
    stems_mensaje = set(stems)
    temas_detectados = []

    for tema, palabras_clave in reglas['entidades']['tema'].items():
        stems_clave = {
            stemmer.stem(token)
            for palabra in palabras_clave
            for token in normalizar_texto(palabra).split()
        }
        if stems_mensaje & stems_clave:
            temas_detectados.append(tema)

    return {'tema': temas_detectados}

detectar_entidades("Duermo mal y estoy estresado por las tareas")

# 5.- Generar respuesta

Aquí ya se genera una respuesta dependiendo del texto ingresado y si es válido

In [ ]:
def generar_respuesta(mensaje: str | None) -> str:
    nombre_intencion, _ = detectar_intencion(mensaje)
    entidades = detectar_entidades(mensaje)
    
    if nombre_intencion == 'fallback':
        return reglas['fallback'][0]

    intencion = next(
        intencion
        for intencion in reglas['intenciones']
        if intencion['nombre'] == nombre_intencion
    )
    respuesta = intencion['respuestas'][0]
    return respuesta.format(**reglas['datos_institucionales'])

generar_respuesta("En donde me puedo comunicar?")